In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Optional


@dataclass
class Llama3Config:
    dim: int = 4096
    n_layers: int = 32
    n_heads: int = 32
    n_kv_heads: int = 8
    vocab_size: int = 128256
    multiple_of: int = 1024
    ffn_dim_multiplier: float = 1.3
    norm_eps: float = 1e-05
    rope_theta: float = 500000.0
    max_seq_len: int = 8192


In [ ]:
def rotate_half(x: torch.Tensor):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]

    return torch.cat([-x2, x1],dim=-1)

In [ ]:
class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, rope_theta):  # 修正了参数名拼写
        super().__init__()

        # 1. 修正 theta_i 计算：分母指数应该是 2i/d，步长为 2，长度为 head_dim // 2
        # 原代码 arange 到 max_seq_len 是错误的，应该是 head_dim
        inv_freq = 1.0 / (
            rope_theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        # 2. 生成频率矩阵 (max_seq_len, head_dim // 2)
        t = torch.arange(max_seq_len, device=inv_freq.device).float()
        freqs = torch.outer(t, inv_freq)  # 形状: (max_seq_len, head_dim // 2)

        # 3. 拼接成 (max_seq_len, head_dim)
        # Llama 3 常用做法是 cat，匹配 rotate_half 的逻辑
        emb = torch.cat([freqs, freqs], dim=-1)

        # 4. 注册缓存，形状为 (1, 1, max_seq_len, head_dim)
        self.register_buffer("cos_cache", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin_cache", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, q: torch.Tensor, k: torch.Tensor):
        # 获取当前输入的序列长度 (seq_len)
        seq_len = q.shape[2]


        cos = self.cos_cache[:, :, :seq_len, :]
        sin = self.sin_cache[:, :, :seq_len, :]

        q_emb = (q * cos) + (rotate_half(q) * sin)
        k_emb = (k * cos) + (rotate_half(k) * sin)

        return q_emb, k_emb


In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim, multiple_of, ffn_dim_multiplier):
        super().__init__()
        # 1. 计算隐藏层维度 (这是 Llama 官方的计算逻辑)
        hidden_dim = int(2 * (dim * 4) / 3)
        if ffn_dim_multiplier is not None:
            hidden_dim = int(ffn_dim_multiplier * hidden_dim)
        # 确保维度是 multiple_of 的倍数
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)

        # 2. 定义层，注意参数名对应
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.v = nn.Linear(dim, hidden_dim, bias=False)  # Llama 官方有时叫 w3
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # 计算均方根并归一化
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight


In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """处理 GQA 的 KV 广播"""
    bs, n_kv_heads, slen, head_dim = x.shape
    if n_rep == 1:
        return x
    x = x[:, :, None, :, :].expand(bs, n_kv_heads, n_rep, slen, head_dim)
    return x.reshape(bs, n_kv_heads * n_rep, slen, head_dim)


class Attention(nn.Module):
    def __init__(self, args, rope: RoPE):
        super().__init__()
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_kv_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.head_dim = args.dim // args.n_heads
        self.d_model = args.dim
        
        self.wq = nn.Linear(self.d_model, self.head_dim * self.n_heads, bias=False)
        self.wk = nn.Linear(self.d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(self.d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(self.head_dim * self.n_heads, self.d_model, bias=False)

        self.rope = rope  # 共享传入的 RoPE 实例

    def forward(self, x: torch.Tensor):
        bsz, seqlen, _ = x.shape

        # 1. 线性映射
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # 2. Reshape 为 (bs, n_heads, seqlen, head_dim) 以便做 Attention
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim).transpose(1, 2)
        xk = xk.view(bsz, seqlen, self.n_kv_heads, self.head_dim).transpose(1, 2)
        xv = xv.view(bsz, seqlen, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # 3. 注入位置编码 RoPE
        xq, xk = self.rope(xq, xk)

        # 4. GQA KV 广播: 将 n_kv_heads 复制对齐到 n_heads
        xk = repeat_kv(xk, self.n_rep)
        xv = repeat_kv(xv, self.n_rep)

        # 5. 计算 Attention (利用 PyTorch 优化的 SDPA 并且自带因果掩码)
        # 注意: is_causal=True 自动处理自回归推理的 Masking
        output = F.scaled_dot_product_attention(xq, xk, xv, is_causal=True)

        # 6. 还原形状并投影输出
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.wo(output)


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, args, rope: RoPE):
        super().__init__()
        self.attention = Attention(args, rope)
        self.feed_forward = SwiGLU(
            dim=args.dim,
            multiple_of=args.multiple_of,
            ffn_dim_multiplier=args.ffn_dim_multiplier,
        )
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)

    def forward(self, x: torch.Tensor):
        # Pre-Norm 架构
        h = x + self.attention(self.attention_norm(x))
        out = h + self.feed_forward(self.ffn_norm(h))
        return out


In [ ]:
class Llama3Model(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.vocab_size = args.vocab_size
        self.n_layers = args.n_layers

        # 词嵌入
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)

        # 初始化全局 RoPE (注意：Llama 3 的 base 变成了 500000.0)
        head_dim = args.dim // args.n_heads
        self.rope = RoPE(
            head_dim=head_dim, max_seq_len=8192, rope_theta=args.rope_theta
        )

        # Transformer 堆叠
        self.layers = nn.ModuleList(
            [TransformerBlock(args, self.rope) for _ in range(args.n_layers)]
        )

        # 输出层
        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

    def forward(self, tokens: torch.Tensor):
        # 数据流: Tokens -> Embeddings -> Layers -> Norm -> LM Head
        h = self.tok_embeddings(tokens)

        for layer in self.layers:
            h = layer(h)

        h = self.norm(h)
        logits = self.output(h)
        return logits


In [ ]:
if __name__ == "__main__":

    print("⏳ 初始化配置和模型 ...")
    config = Llama3Config()

    device = "cuda" if torch.cuda.is_available() else 'cpu'
    target_dtype = torch.bfloat16
    # 实例化模型
    with torch.device(device):
        model = Llama3Model(config)
        model.to(dtype=target_dtype)
    model.eval()

    # 模拟输入数据: Batch_size=2, Sequence_length=15
    bsz = 2
    seqlen = 15
    dummy_input = torch.randint(0, config.vocab_size, (bsz, seqlen),device=device)

    print(f"📥 输入 Tensor 形状: {dummy_input.shape} (Batch Size, Sequence Length)")

    # 测试前向传播
    with torch.no_grad():
        out_logits = model(dummy_input)

    print(
        f"📤 输出 Logits 形状: {out_logits.shape} (Batch Size, Sequence Length, Vocab Size)"
    )
    print("✅ 配置加载成功，数据流完美跑通！")
